# GlyphMatics Nemotron Submission — v11 Forced Salvage Rebuild

**Base:** uploaded `glyphmatics-1.ipynb` (v8 PairFold RowGuard Transport).

**Added components:**
- deterministic hybrid selector
- tailguard / 100% restore candidate
- dual-pair energy-cap candidate
- deterministic salvage rebuild candidate
- closest-match residual salvage
- pair-of-pairs residual basis scoring
- stronger build + zip validation + output-path FileExists guard
- fallback profile switches

v10 fix: the build output folder is removed and intentionally not recreated before `build_lora_adapter`, because Tinker raises `FileExistsError` if `output_path` already exists.\n\nThis notebook stays competition-oriented and Kaggle-ready while preserving the uploaded PairFold row-guard transport as a selectable candidate inside the hybrid merge patch.


## v11 change

The v10 public result stayed at `0.86`, which means the hybrid guard likely selected the same saturated tailguard path.  
v11 disables that escape route by default and forces the deterministic salvage-rebuild candidate so the run tests a genuinely different transform.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import zipfile
import importlib.util
import inspect
import time

print("Python:", sys.version)
print("Working dir:", Path.cwd())

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle. /kaggle/input was not found.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

# Deterministic/noise-control defaults.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONHASHSEED", "918")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Core rank contract. Prior board saturation was around rank-32 variants.
os.environ.setdefault("FORCED_FUSED_RANK", "32")

# Hybrid mode: choose among old saturated baselines and new deterministic salvage rebuild.
os.environ.setdefault("GLYPHMATIC_PATCH_MODE", "salvage_guarded")  # hybrid_guarded | salvage_guarded | tailguard | dual_pair | baseline
os.environ.setdefault("GLYPHMATIC_FORCE_CANDIDATE", "salvage")           # empty | tailguard | dual_pair | salvage

# Tailguard / 100% restore source behavior.
os.environ.setdefault("GLYPHMATIC_RESTORE_RATIO", "1.00")
os.environ.setdefault("GLYPHMATIC_HEAD_RANK", "24")
os.environ.setdefault("GLYPHMATIC_TAIL_FACTOR", "0.88")
os.environ.setdefault("GLYPHMATIC_SCALE_HARD_CAP", "0")

# Dual-pair energy-cap source behavior.
os.environ.setdefault("SVD_ENERGY_GAIN_CAP", "1.20")
os.environ.setdefault("DUAL_PAIR_ENABLED", "1")
os.environ.setdefault("DUAL_PAIR_SPLIT", "0.62,0.58")

# Deterministic salvage rebuild behavior.
os.environ.setdefault("GLYPHMATIC_SALVAGE_ENABLE", "1")
os.environ.setdefault("GLYPHMATIC_SALVAGE_FORCE", "1")
os.environ.setdefault("GLYPHMATIC_SALVAGE_HEAD_RANK", os.environ["GLYPHMATIC_HEAD_RANK"])
os.environ.setdefault("GLYPHMATIC_SALVAGE_LOCAL_RANK", "3")
os.environ.setdefault("GLYPHMATIC_SALVAGE_PAIR_RANK", "3")
os.environ.setdefault("GLYPHMATIC_CLOSEST_MATCH_ENABLE", "1")
os.environ.setdefault("GLYPHMATIC_CLOSEST_MATCH_RANK", "4")
os.environ.setdefault("GLYPHMATIC_SALVAGE_MARGIN", "0.0000")

# Balanced candidate gate weights.
os.environ.setdefault("GLYPHMATIC_BLOCK_SCORE_WEIGHT", "0.35")
os.environ.setdefault("GLYPHMATIC_ENERGY_SCORE_WEIGHT", "0.15")
os.environ.setdefault("GLYPHMATIC_GLOBAL_DRIFT_LIMIT", "0.010")

print("\n[Inputs]")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p)

print("\n[GlyphMatics env]")
interesting = []
for k in sorted(os.environ):
    if k.startswith("GLYPHMATIC") or k in {"FORCED_FUSED_RANK", "SVD_ENERGY_GAIN_CAP", "DUAL_PAIR_ENABLED", "DUAL_PAIR_SPLIT"}:
        interesting.append(k)
for k in interesting:
    print(f" {k}={os.environ[k]}")


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models

[GlyphMatics env]
 DUAL_PAIR_ENABLED=1
 DUAL_PAIR_SPLIT=0.62,0.58
 FORCED_FUSED_RANK=32
 GLYPHMATIC_BLOCK_SCORE_WEIGHT=0.35
 GLYPHMATIC_CLOSEST_MATCH_ENABLE=1
 GLYPHMATIC_CLOSEST_MATCH_RANK=4
 GLYPHMATIC_ENERGY_SCORE_WEIGHT=0.15
 GLYPHMATIC_FORCE_CANDIDATE=salvage
 GLYPHMATIC_GLOBAL_DRIFT_LIMIT=0.010
 GLYPHMATIC_HEAD_RANK=24
 GLYPHMATIC_PATCH_MODE=salvage_guarded
 GLYPHMATIC_RESTORE_RATIO=1.00
 GLYPHMATIC_SALVAGE_ENABLE=1
 GLYPHMATIC_SALVAGE_FORCE=1
 GLYPHMATIC_SALVAGE_HEAD_RANK=24
 GLYPHMATIC_SALVAGE_LOCAL_RANK=3
 GLYPHMATIC_SALVAGE_MARGIN=0.0000
 GLYPHMATIC_SALVAGE_PAIR_RANK=3
 GLYPHMATIC_SCALE_HARD_CAP=0
 GLYPHMATIC_TAIL_FACTOR=0.88
 SVD_ENERGY_GAIN_CAP=1.20


## 1. Install / load Tinker from local wheels

This scans Kaggle inputs for local wheels and installs offline with `--no-index`.

If you already attached a wheel dataset, this cell should find it automatically. You can also set `WHEEL_DIR` manually before running.


In [2]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import importlib.metadata as md

def list_wheel_dirs():
    rows = []
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]
    for root in roots:
        if not root.exists():
            continue
        # rglob can be expensive, but the explicit wheel scan is what prevents ModuleNotFoundError.
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            try:
                wheels = sorted(d.glob("*.whl"))
            except PermissionError:
                continue
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def score_tinker_dir(names):
    low = " ".join(n.lower() for n in names)
    score = 0
    for token in ["tinker_cookbook", "tinker-cookbook"]:
        if token in low:
            score += 6
    for token in ["tinker_", "tinker-"]:
        if token in low:
            score += 3
    if "chz" in low:
        score += 2
    return score

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        score = score_tinker_dir(names)
        if score:
            candidates.append((score, len(names), str(d), d, names))
    candidates.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    return candidates[0] if candidates else None

print("[Tinker] scanning Kaggle inputs for local wheels...")
wheel_dirs = list_wheel_dirs()
for d, names in wheel_dirs:
    interesting = [n for n in names if ("tinker" in n.lower() or "chz" in n.lower())]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting[:80]:
            print(" -", name)
        if len(interesting) > 80:
            print(f" ... {len(interesting)-80} more")

force_install = os.environ.get("FORCE_TINKER_INSTALL", "0") == "1"

if importlib.util.find_spec("tinker_cookbook") is not None and not force_install:
    print("[Tinker] tinker_cookbook already installed; skipping wheel install")
else:
    explicit = os.environ.get("WHEEL_DIR")
    candidate = None

    if explicit and Path(explicit).exists():
        candidate = (999, 0, str(Path(explicit)), Path(explicit), [p.name for p in Path(explicit).glob("*.whl")])
    else:
        candidate = find_tinker_wheelhouse()

    if candidate is None:
        raise FileNotFoundError(
            "Could not find local tinker wheelhouse. Attach a Kaggle input containing "
            "tinker-cookbook/tinker/chz wheels, or set WHEEL_DIR to that folder. "
            "Run this install cell BEFORE any cell importing tinker_cookbook."
        )

    _, _, _, wheel_dir, names = candidate
    print("[Tinker] selected wheel_dir:", wheel_dir)

    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={wheel_dir}",
        "tinker-cookbook",
        "tinker",
    ]
    print("[Tinker] pip:", " ".join(cmd))
    subprocess.run(cmd, check=True)

import tinker_cookbook
from tinker_cookbook import weights

print("[Tinker] ready:", tinker_cookbook.__file__)
for package in ["tinker-cookbook", "tinker"]:
    try:
        print(f"[Tinker] {package} version:", md.version(package))
    except Exception as e:
        print(f"[Tinker] {package} version unavailable:", repr(e))

print("[Tinker] has build_lora_adapter:", hasattr(weights, "build_lora_adapter"))
if not hasattr(weights, "build_lora_adapter"):
    raise RuntimeError("tinker_cookbook.weights.build_lora_adapter not found")

print("[Tinker] build_lora_adapter signature:", inspect.signature(weights.build_lora_adapter))


[Tinker] scanning Kaggle inputs for local wheels...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
[Tinker] pip: /usr/bin/python3 -m pip install --no-index --find-links=/kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse tinker-cookbook tinker
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl (from tinker-cookbook)
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker] tinker-cookbook 

## 2. Detect base model and adapter paths

The path finder prefers the known Kaggle model layout, then falls back to recursive detection across attached inputs.


In [3]:
from pathlib import Path
import json
import os

def find_first_existing(paths):
    for p in paths:
        q = Path(p)
        if q.exists():
            return q
    return None

def score_path(path: Path, tokens):
    s = str(path).lower()
    score = 0
    for weight, token in tokens:
        if token in s:
            score += weight
    return score

def find_adapter_path():
    explicit = os.environ.get("ADAPTER_PATH")
    if explicit and Path(explicit).exists():
        return Path(explicit)

    candidates = [
        "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
        "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
        "/kaggle/input/nemotron-adapter/transformers/default/20",
        "/kaggle/input/nemotron-adapter",
    ]
    found = find_first_existing(candidates)
    if found is not None:
        return found

    roots = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        root = cfg.parent
        has_weights = (
            (root / "adapter_model.safetensors").exists()
            or (root / "adapter_model.bin").exists()
            or any(root.glob("*adapter*.safetensors"))
            or any(root.glob("*.safetensors"))
        )
        if has_weights:
            roots.append(root)

    if not roots:
        return None

    roots.sort(key=lambda p: score_path(p, [(8, "huikang"), (6, "nemotron-adapter"), (4, "adapter"), (2, "transformers")]), reverse=True)
    return roots[0]

def find_base_model_path():
    explicit = os.environ.get("BASE_MODEL_PATH")
    if explicit and Path(explicit).exists():
        return Path(explicit)

    candidates = [
        "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
    ]
    found = find_first_existing(candidates)
    if found is not None:
        return found

    roots = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        root = cfg.parent
        # Ignore adapter configs and tiny config-only directories.
        if (root / "adapter_config.json").exists():
            continue
        has_weights = (
            (root / "model.safetensors.index.json").exists()
            or (root / "pytorch_model.bin.index.json").exists()
            or any(root.glob("model-*.safetensors"))
            or any(root.glob("*.safetensors"))
            or any(root.glob("pytorch_model*.bin"))
        )
        if has_weights:
            roots.append(root)

    if not roots:
        return None

    roots.sort(key=lambda p: score_path(p, [(10, "nemotron"), (6, "30b"), (5, "a3b"), (4, "bf16"), (2, "metric")]), reverse=True)
    return roots[0]

ADAPTER_PATH = find_adapter_path()
BASE_MODEL_PATH = find_base_model_path()

if ADAPTER_PATH is None:
    print("[Adapter candidates debug]")
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        print(" -", cfg.parent)
    raise FileNotFoundError(
        "Adapter path not found. Attach huikang/nemotron-adapter or set ADAPTER_PATH."
    )

if BASE_MODEL_PATH is None:
    print("[Base model candidates debug]")
    for cfg in Path("/kaggle/input").rglob("config.json"):
        print(" -", cfg.parent)
    raise FileNotFoundError(
        "Nemotron base model path not found. Attach nemotron-3-nano-30b-a3b-bf16 or set BASE_MODEL_PATH."
    )

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

print("\n[Adapter files]")
for p in sorted(ADAPTER_PATH.iterdir()):
    print(" -", p.name)

print("\n[Base model key files]")
for name in ["config.json", "model.safetensors.index.json", "pytorch_model.bin.index.json", "tokenizer.json", "tokenizer_config.json"]:
    p = BASE_MODEL_PATH / name
    print(f" - {name}:", p.exists())

# Persist path metadata for final manifest.
PATHS_MANIFEST = {
    "adapter_path": str(ADAPTER_PATH),
    "base_model_path": str(BASE_MODEL_PATH),
}


[Paths] ADAPTER_PATH: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Paths] BASE_MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1

[Adapter files]
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - checkpoint_complete

[Base model key files]
 - config.json: True
 - model.safetensors.index.json: True
 - pytorch_model.bin.index.json: False
 - tokenizer.json: True
 - tokenizer_config.json: True


## 3. Apply GlyphMatics v9 PairFold hybrid fused-projection transport patch

This preserves the uploaded PairFold RowGuard transport as a candidate and adds the new hybrid selector, deterministic salvage rebuild, closest-match residual salvage, tailguard 100% restore, and dual-pair energy-cap candidates.


In [4]:
from __future__ import annotations

from collections import Counter
from typing import Any, Dict, List, Tuple
from pathlib import Path
import json
import os
import hashlib
import math

import torch
import tinker_cookbook.weights._adapter as A

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# -----------------------------------------------------------------------------
# Competition knobs
# -----------------------------------------------------------------------------
FORCED_FUSED_RANK = int(os.environ.get("FORCED_FUSED_RANK", "32"))
SVD_ENERGY_GAIN_CAP = float(os.environ.get("SVD_ENERGY_GAIN_CAP", "1.19"))
GAIN_DISTRIBUTION = "pairfold_residual_matched_balanced_rowguard"

ROW_NORM_GUARD_ENABLED = os.environ.get("ROW_NORM_GUARD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
ROW_NORM_GAIN_CAP = float(os.environ.get("ROW_NORM_GAIN_CAP", "1.08"))

PAIRFOLD_ENABLED = os.environ.get("PAIRFOLD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
PAIRFOLD_PAIR_WIDTH = int(os.environ.get("PAIRFOLD_PAIR_WIDTH", "2"))
PAIRFOLD_TAIL_MAX = int(os.environ.get("PAIRFOLD_TAIL_MAX", "96"))
PAIRFOLD_MIN_SIM = float(os.environ.get("PAIRFOLD_MIN_SIM", "0.66"))
PAIRFOLD_MAX_GAIN_DELTA = float(os.environ.get("PAIRFOLD_MAX_GAIN_DELTA", "0.022"))
PAIRFOLD_VECTOR_BLEND = float(os.environ.get("PAIRFOLD_VECTOR_BLEND", "0.025"))
PAIRFOLD_DECAY = float(os.environ.get("PAIRFOLD_DECAY", "0.985"))
PAIRFOLD_COL_BINS = int(os.environ.get("PAIRFOLD_COL_BINS", "8"))

DUAL_PAIR_ENABLED = os.environ.get("DUAL_PAIR_ENABLED", "0").strip().lower() not in {"0", "false", "no", "off"}
DUAL_PAIR_SPLIT_RAW = os.environ.get("DUAL_PAIR_SPLIT", "0.62,0.58")

if FORCED_FUSED_RANK <= 0:
    raise ValueError("FORCED_FUSED_RANK must be positive")
if not (1.0 <= SVD_ENERGY_GAIN_CAP <= 1.25):
    raise ValueError("SVD_ENERGY_GAIN_CAP must stay inside [1.0, 1.25]")
if not (1.0 <= ROW_NORM_GAIN_CAP <= 1.25):
    raise ValueError("ROW_NORM_GAIN_CAP must stay inside [1.0, 1.25]")
if PAIRFOLD_PAIR_WIDTH <= 0:
    raise ValueError("PAIRFOLD_PAIR_WIDTH must be positive")
if PAIRFOLD_TAIL_MAX < 0:
    raise ValueError("PAIRFOLD_TAIL_MAX must be non-negative")
if not (0.0 <= PAIRFOLD_MIN_SIM <= 1.0):
    raise ValueError("PAIRFOLD_MIN_SIM must be inside [0, 1]")
if not (0.0 <= PAIRFOLD_MAX_GAIN_DELTA <= 0.10):
    raise ValueError("PAIRFOLD_MAX_GAIN_DELTA must be inside [0, 0.10]")
if not (0.0 <= PAIRFOLD_VECTOR_BLEND <= 0.20):
    raise ValueError("PAIRFOLD_VECTOR_BLEND must be inside [0, 0.20]")
if not (0.50 <= PAIRFOLD_DECAY <= 1.0):
    raise ValueError("PAIRFOLD_DECAY must be inside [0.50, 1.0]")
if not (1 <= PAIRFOLD_COL_BINS <= 64):
    raise ValueError("PAIRFOLD_COL_BINS must be inside [1, 64]")

def _parse_pair_split(raw: str):
    vals = []
    for piece in raw.replace(";", ",").split(","):
        piece = piece.strip()
        if piece:
            vals.append(float(piece))
    if len(vals) != 2:
        raise ValueError("DUAL_PAIR_SPLIT must contain exactly two numbers, e.g. '0.62,0.58'")
    if not all(v > 0 for v in vals):
        raise ValueError("DUAL_PAIR_SPLIT values must be positive")
    return vals

DUAL_PAIR_SPLIT = _parse_pair_split(DUAL_PAIR_SPLIT_RAW)

# -----------------------------------------------------------------------------
# Ledger
# -----------------------------------------------------------------------------
class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        beta = beta or {}
        gamma = gamma or {}
        basis = json.dumps(
            {"alpha": alpha, "src": str(src), "dst": str(dst), "op": str(op), "beta": beta, "gamma": gamma},
            sort_keys=True,
            default=str,
        )
        event = {
            "alpha": str(alpha),
            "source": str(src),
            "dest": str(dst),
            "op": str(op),
            "beta": beta,
            "gamma": gamma,
            "verification_hash": hashlib.sha256(basis.encode("utf-8")).hexdigest()[:16],
        }
        self.events.append(event)
        self.counts[(event["alpha"], event["op"])] += 1

    def markdown(self) -> str:
        lines = [
            "# GlyphMatics Transport Ledger",
            "",
            "Generated during tinker-cookbook adapter conversion.",
            "",
            "## Submission configuration",
            "",
            f"- `FORCED_FUSED_RANK`: `{FORCED_FUSED_RANK}`",
            f"- `SVD_ENERGY_GAIN_CAP`: `{SVD_ENERGY_GAIN_CAP}`",
            f"- `GAIN_DISTRIBUTION`: `{GAIN_DISTRIBUTION}`",
            f"- `PAIRFOLD_ENABLED`: `{PAIRFOLD_ENABLED}`",
            f"- `PAIRFOLD_PAIR_WIDTH`: `{PAIRFOLD_PAIR_WIDTH}`",
            f"- `PAIRFOLD_TAIL_MAX`: `{PAIRFOLD_TAIL_MAX}`",
            f"- `PAIRFOLD_MIN_SIM`: `{PAIRFOLD_MIN_SIM}`",
            f"- `PAIRFOLD_MAX_GAIN_DELTA`: `{PAIRFOLD_MAX_GAIN_DELTA}`",
            f"- `PAIRFOLD_VECTOR_BLEND`: `{PAIRFOLD_VECTOR_BLEND}`",
            f"- `PAIRFOLD_DECAY`: `{PAIRFOLD_DECAY}`",
            f"- `PAIRFOLD_COL_BINS`: `{PAIRFOLD_COL_BINS}`",
            f"- `ROW_NORM_GUARD_ENABLED`: `{ROW_NORM_GUARD_ENABLED}`",
            f"- `ROW_NORM_GAIN_CAP`: `{ROW_NORM_GAIN_CAP}`",
            f"- `DUAL_PAIR_ENABLED`: `{DUAL_PAIR_ENABLED}`",
            f"- `DUAL_PAIR_SPLIT`: `{DUAL_PAIR_SPLIT}`",
            "",
            "## Event summary",
            "",
            "| alpha | op | count |",
            "|---|---|---:|",
        ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")

        lines += [
            "",
            "## First 80 events",
            "",
            "| # | alpha | op | source | destination | gamma | hash |",
            "|---:|---|---|---|---|---|---|",
        ]
        for i, event in enumerate(self.events[:80], 1):
            gamma = json.dumps(event["gamma"], sort_keys=True, default=str)
            lines.append(
                f"| {i} | `{event['alpha']}` | `{event['op']}` | "
                f"`{event['source']}` | `{event['dest']}` | `{gamma}` | `{event['verification_hash']}` |"
            )
        return "\n".join(lines) + "\n"

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger] {alpha}:{op}={count}")

GLYPH_LEDGER = GlyphmaticTransportLedger()

# -----------------------------------------------------------------------------
# Numeric helpers
# -----------------------------------------------------------------------------
def _safe_unit(x: torch.Tensor, dim=None, eps: float = 1e-12):
    if dim is None:
        return x / torch.linalg.vector_norm(x).clamp_min(eps)
    return x / torch.linalg.vector_norm(x, dim=dim, keepdim=True).clamp_min(eps)

def _make_even_blocks(length: int, count: int, prefix: str):
    count = max(1, min(int(count), int(length)))
    blocks = []
    for i in range(count):
        start = int(round(i * length / count))
        end = int(round((i + 1) * length / count))
        if end > start:
            blocks.append((start, end, f"{prefix}{i}"))
    return blocks

def _make_row_blocks(row_count: int, component_slices=None):
    blocks = []
    if component_slices:
        for row_start, row_end, _rank, name in component_slices:
            row_start = max(0, min(int(row_start), int(row_count)))
            row_end = max(row_start, min(int(row_end), int(row_count)))
            if row_end > row_start:
                blocks.append((row_start, row_end, str(name)))
    covered = sum(e - s for s, e, _ in blocks)
    if not blocks or covered < row_count:
        # Fallback also covers non-component rows in unusual fused layouts.
        blocks = _make_even_blocks(row_count, min(8, row_count), "rowbin")
    return blocks

def _block_energy(vec: torch.Tensor, blocks):
    vals = []
    vec = vec.float()
    sq = vec * vec
    for start, end, _name in blocks:
        vals.append(sq[start:end].sum())
    out = torch.stack(vals) if vals else torch.ones(1, device=vec.device, dtype=torch.float32)
    return _safe_unit(out.float())

def _direction_signature(u_col: torch.Tensor, vh_row: torch.Tensor, row_blocks, col_bins: int):
    """
    Local behavior signature for residual matching.

    Raw SVD vectors are orthogonal globally, so direct cosine is not useful.
    This signature compares where a direction spends energy by fused output rows
    and input-column bins. Tail directions are folded into survivor pairs with
    similar local behavior.
    """
    col_blocks = _make_even_blocks(int(vh_row.numel()), max(1, min(col_bins, int(vh_row.numel()))), "colbin")
    row_sig = _block_energy(u_col, row_blocks)
    col_sig = _block_energy(vh_row, col_blocks)
    u_abs = u_col.float().abs()
    v_abs = vh_row.float().abs()
    moments = torch.tensor(
        [
            float(u_abs.max().detach().cpu()),
            float(v_abs.max().detach().cpu()),
            float(u_abs.mean().detach().cpu()),
            float(v_abs.mean().detach().cpu()),
        ],
        device=u_col.device,
        dtype=torch.float32,
    )
    return _safe_unit(torch.cat([row_sig, col_sig, _safe_unit(moments)]).float())

def _rank_pair_groups(rank: int, pair_width: int):
    groups = []
    start = 0
    while start < rank:
        end = min(rank, start + pair_width)
        groups.append(list(range(start, end)))
        start = end
    return groups

def _dual_lane_gain_vector(*, singular_values: torch.Tensor, raw_gain: torch.Tensor, rank: int):
    """
    Stable two-lane option retained for A/B tests.

    By default DUAL_PAIR_ENABLED=0 because PairFold is now the primary transport.
    """
    base_gain = torch.clamp(raw_gain, min=1.0, max=SVD_ENERGY_GAIN_CAP)
    gain_vec = torch.full_like(singular_values, fill_value=float(base_gain))

    if not DUAL_PAIR_ENABLED or rank < 2:
        return gain_vec, {
            "mode": "single_lane_before_pairfold",
            "raw_energy_gain": float(raw_gain.detach().cpu()),
            "global_energy_gain": float(base_gain.detach().cpu()),
            "lane_ranks": [int(rank)],
            "lane_caps": [float(SVD_ENERGY_GAIN_CAP)],
            "lane_gains": [float(base_gain.detach().cpu())],
        }

    first = rank // 2
    second = rank - first
    pair_mean = sum(DUAL_PAIR_SPLIT) / 2.0
    cap_excess = max(SVD_ENERGY_GAIN_CAP - 1.0, 0.0)

    lane_caps = [
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[0] / pair_mean)),
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[1] / pair_mean)),
    ]

    lane_gain_0 = torch.clamp(raw_gain, min=1.0, max=lane_caps[0])
    lane_gain_1 = torch.clamp(raw_gain, min=1.0, max=lane_caps[1])

    gain_vec[:first] = lane_gain_0
    gain_vec[first:first + second] = lane_gain_1

    return gain_vec, {
        "mode": "dual_pair_before_pairfold",
        "raw_energy_gain": float(raw_gain.detach().cpu()),
        "global_energy_gain_cap": float(SVD_ENERGY_GAIN_CAP),
        "dual_pair_split": [float(x) for x in DUAL_PAIR_SPLIT],
        "lane_ranks": [int(first), int(second)],
        "lane_caps": [float(x) for x in lane_caps],
        "lane_gains": [float(lane_gain_0.detach().cpu()), float(lane_gain_1.detach().cpu())],
    }

def _pairfold_transport(U: torch.Tensor, S: torch.Tensor, Vh: torch.Tensor, rank: int, gain_vec: torch.Tensor, row_blocks):
    """
    Residual-matched rank-pair compression.

    The discarded SVD tail is not directly retained as extra rank. Instead:
    1. Each kept direction gets a local behavior signature.
    2. Adjacent kept directions are grouped into rank pairs.
    3. Tail directions are matched to the closest surviving pair by signature.
    4. Restoration gain is redistributed toward pairs that absorbed similar tail.
    5. A tiny capped orientation blend can nudge the survivor pair toward the tail.

    Global gain is energy-renormalized back near the incoming target so this
    changes transport shape more than total volume.
    """
    device = U.device
    rank = int(rank)
    tail_available = max(0, int(S.numel()) - rank)
    if (not PAIRFOLD_ENABLED) or rank <= 0 or tail_available <= 0 or PAIRFOLD_TAIL_MAX <= 0:
        return U[:, :rank].contiguous(), Vh[:rank, :].contiguous(), gain_vec.contiguous(), {
            "pairfold_enabled": bool(PAIRFOLD_ENABLED),
            "pairfold_mode": "disabled_or_no_tail",
            "pairfold_assignments": 0,
            "pairfold_tail_used": 0,
        }

    U_k = U[:, :rank].clone()
    Vh_k = Vh[:rank, :].clone()
    S_k = S[:rank]
    gain_in = gain_vec.clone()

    head_sigs = torch.stack([
        _direction_signature(U[:, i], Vh[i, :], row_blocks, PAIRFOLD_COL_BINS)
        for i in range(rank)
    ])

    groups = _rank_pair_groups(rank, PAIRFOLD_PAIR_WIDTH)
    group_sigs = []
    for group in groups:
        idx = torch.tensor(group, device=device, dtype=torch.long)
        weights = S_k[idx].float().clamp_min(1e-12)
        weights = weights / weights.sum().clamp_min(1e-12)
        sig = (head_sigs[idx] * weights.unsqueeze(1)).sum(dim=0)
        group_sigs.append(_safe_unit(sig))
    group_sigs = torch.stack(group_sigs)

    pair_priority = torch.zeros(len(groups), device=device, dtype=torch.float32)
    head_priority = torch.zeros(rank, device=device, dtype=torch.float32)
    u_acc = torch.zeros_like(U_k.float())
    v_acc = torch.zeros_like(Vh_k.float())

    tail_count = min(tail_available, int(PAIRFOLD_TAIL_MAX))
    assignments = 0
    sim_sum = 0.0
    max_sim = 0.0

    for local_j in range(tail_count):
        j = rank + local_j
        tail_sig = _direction_signature(U[:, j], Vh[j, :], row_blocks, PAIRFOLD_COL_BINS)
        sims = group_sigs @ tail_sig
        best_group = int(torch.argmax(sims).detach().cpu())
        best_sim = float(sims[best_group].detach().cpu())
        if best_sim < PAIRFOLD_MIN_SIM:
            continue

        decay = float(PAIRFOLD_DECAY ** local_j)
        # Priority is relative, not raw energy injection. It tells where the
        # already-safe restoration should be placed.
        rel_tail = float((S[j] / S_k.mean().clamp_min(1e-12)).detach().cpu())
        priority = max(0.0, best_sim * decay * rel_tail)
        if priority <= 0:
            continue

        assignments += 1
        sim_sum += best_sim
        max_sim = max(max_sim, best_sim)
        pair_priority[best_group] += priority

        group = groups[best_group]
        idx = torch.tensor(group, device=device, dtype=torch.long)
        local_sims = (head_sigs[idx] @ tail_sig).clamp_min(0.0)
        local_weights = local_sims + S_k[idx].float() / S_k[idx].float().sum().clamp_min(1e-12)
        local_weights = local_weights / local_weights.sum().clamp_min(1e-12)

        for pos, head_idx in enumerate(group):
            head_priority[head_idx] += priority * float(local_weights[pos].detach().cpu())
            if PAIRFOLD_VECTOR_BLEND > 0:
                # Tiny orientation fold. Capped per tail and scaled by similarity.
                blend = min(
                    PAIRFOLD_VECTOR_BLEND,
                    PAIRFOLD_VECTOR_BLEND * best_sim * rel_tail,
                ) * float(local_weights[pos].detach().cpu())
                u_acc[:, head_idx] += float(blend) * U[:, j].float()
                v_acc[head_idx, :] += float(blend) * Vh[j, :].float()

    if assignments == 0 or float(head_priority.sum().detach().cpu()) <= 0.0:
        return U_k.contiguous(), Vh_k.contiguous(), gain_in.contiguous(), {
            "pairfold_enabled": True,
            "pairfold_mode": "no_similarity_match",
            "pairfold_assignments": 0,
            "pairfold_tail_used": int(tail_count),
            "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        }

    # Rank-pair gain redistribution.
    # Center around 1.0, cap the delta, then energy-renormalize back to the
    # pre-PairFold target. This avoids simply making the adapter louder.
    priority = head_priority / head_priority.mean().clamp_min(1e-12)
    delta = (priority - 1.0).clamp(-1.0, 1.0) * float(PAIRFOLD_MAX_GAIN_DELTA)
    gain_out = gain_in * (1.0 + delta)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    target_energy = torch.sqrt(torch.sum((S_k * gain_in) ** 2)).clamp_min(1e-12)
    current_energy = torch.sqrt(torch.sum((S_k * gain_out) ** 2)).clamp_min(1e-12)
    gain_out = gain_out * (target_energy / current_energy)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    if PAIRFOLD_VECTOR_BLEND > 0:
        U_k = _safe_unit(U_k.float() + u_acc, dim=0).to(U.dtype)
        Vh_k = _safe_unit(Vh_k.float() + v_acc, dim=1).to(Vh.dtype)

    return U_k.contiguous(), Vh_k.contiguous(), gain_out.contiguous(), {
        "pairfold_enabled": True,
        "pairfold_mode": "residual_matched_rank_pairs",
        "pairfold_pair_width": int(PAIRFOLD_PAIR_WIDTH),
        "pairfold_tail_used": int(tail_count),
        "pairfold_assignments": int(assignments),
        "pairfold_assignment_rate": float(assignments / max(1, tail_count)),
        "pairfold_avg_match_sim": float(sim_sum / max(1, assignments)),
        "pairfold_max_match_sim": float(max_sim),
        "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        "pairfold_max_gain_delta": float(PAIRFOLD_MAX_GAIN_DELTA),
        "pairfold_vector_blend": float(PAIRFOLD_VECTOR_BLEND),
        "pairfold_gain_mean": float(gain_out.mean().detach().cpu()),
        "pairfold_gain_min": float(gain_out.min().detach().cpu()),
        "pairfold_gain_max": float(gain_out.max().detach().cpu()),
    }

def _apply_row_norm_guard(B_new: torch.Tensor, A_new: torch.Tensor, delta: torch.Tensor):
    """Clip local row-level overshoot after compression."""
    if not ROW_NORM_GUARD_ENABLED:
        return B_new.contiguous(), {
            "row_norm_guard_enabled": False,
            "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
            "row_clip_fraction": 0.0,
        }

    with torch.no_grad():
        recon = B_new.float() @ A_new.float()
        orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
        new_row = torch.linalg.vector_norm(recon, ord=2, dim=1).clamp_min(1e-12)
        ratio = new_row / orig_row
        max_allowed = orig_row * float(ROW_NORM_GAIN_CAP)
        row_scale = torch.minimum(torch.ones_like(new_row), max_allowed / new_row)
        clipped = row_scale < 0.999
        B_guarded = (B_new.float() * row_scale.unsqueeze(1)).to(B_new.dtype).contiguous()

    return B_guarded, {
        "row_norm_guard_enabled": True,
        "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
        "row_clip_fraction": float(clipped.float().mean().detach().cpu()),
        "row_clip_count": int(clipped.sum().detach().cpu()),
        "row_ratio_mean_before_guard": float(ratio.mean().detach().cpu()),
        "row_ratio_max_before_guard": float(ratio.max().detach().cpu()),
    }

def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int, component_slices=None):
    """
    Compress Delta = B @ A to rank-k with PairFold residual matching.

    The output shape is always `(out_dim, rank)` and `(rank, in_dim)` unless
    the underlying matrix is smaller than the requested rank.
    """
    if B.ndim != 2 or A_mat.ndim != 2:
        raise ValueError(f"Expected 2D LoRA matrices, got B={tuple(B.shape)}, A={tuple(A_mat.shape)}")
    if B.shape[1] != A_mat.shape[0]:
        raise ValueError(f"LoRA inner rank mismatch: B={tuple(B.shape)}, A={tuple(A_mat.shape)}")

    delta = B.float() @ A_mat.float()
    if not torch.isfinite(delta).all():
        raise FloatingPointError("Dense LoRA delta contains non-finite values before compression")

    max_rank = min(delta.shape)
    rank = min(int(rank), int(max_rank))
    if rank <= 0:
        raise ValueError(f"Invalid compression rank {rank} for delta shape {tuple(delta.shape)}")

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)
    raw_gain = full_energy / kept_energy
    gain_vec, gain_stats = _dual_lane_gain_vector(
        singular_values=S_k,
        raw_gain=raw_gain,
        rank=rank,
    )

    row_blocks = _make_row_blocks(int(delta.shape[0]), component_slices=component_slices)
    U_k, Vh_k, gain_vec, pairfold_stats = _pairfold_transport(
        U=U,
        S=S,
        Vh=Vh,
        rank=rank,
        gain_vec=gain_vec,
        row_blocks=row_blocks,
    )

    # Balanced transport: split scale symmetrically across both LoRA factors.
    sroot_balanced = torch.sqrt(S_k * gain_vec)
    B_new = U_k * sroot_balanced.unsqueeze(0)
    A_new = sroot_balanced.unsqueeze(1) * Vh_k
    B_new, row_guard_stats = _apply_row_norm_guard(B_new, A_new, delta)

    restored_energy = torch.sqrt(torch.sum((S_k * gain_vec) ** 2)).clamp_min(1e-12)
    recon = B_new.float() @ A_new.float()
    residual_energy = torch.linalg.vector_norm((delta - recon).float()).clamp_min(1e-12)

    if not torch.isfinite(B_new).all() or not torch.isfinite(A_new).all():
        raise FloatingPointError("Compressed LoRA factors contain non-finite values")

    stats = {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "delta_shape": [int(delta.shape[0]), int(delta.shape[1])],
        "singular_mass_kept": float((S_k.sum() / total_mass).detach().cpu()),
        "energy_kept_ratio": float((kept_energy / full_energy).detach().cpu()),
        "energy_after_gain_ratio": float((restored_energy / full_energy).detach().cpu()),
        "reconstruction_residual_ratio": float((residual_energy / torch.linalg.vector_norm(delta).clamp_min(1e-12)).detach().cpu()),
        "gain_distribution": GAIN_DISTRIBUTION,
        "row_block_count": int(len(row_blocks)),
        "row_blocks": [{"name": name, "rows": [int(start), int(end)]} for start, end, name in row_blocks],
        **gain_stats,
        **pairfold_stats,
        **row_guard_stats,
    }

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), stats

# -----------------------------------------------------------------------------
# Hybrid candidate-selection knobs
# -----------------------------------------------------------------------------
GLYPHMATIC_PATCH_MODE = os.environ.get("GLYPHMATIC_PATCH_MODE", "hybrid_guarded").lower().strip()
GLYPHMATIC_FORCE_CANDIDATE = os.environ.get("GLYPHMATIC_FORCE_CANDIDATE", "").lower().strip()

GLYPHMATIC_RESTORE_RATIO = float(os.environ.get("GLYPHMATIC_RESTORE_RATIO", "1.00"))
GLYPHMATIC_HEAD_RANK = int(os.environ.get("GLYPHMATIC_HEAD_RANK", "24"))
GLYPHMATIC_TAIL_FACTOR = float(os.environ.get("GLYPHMATIC_TAIL_FACTOR", "0.88"))
GLYPHMATIC_SCALE_HARD_CAP = float(os.environ.get("GLYPHMATIC_SCALE_HARD_CAP", "0"))

GLYPHMATIC_SALVAGE_ENABLE = int(os.environ.get("GLYPHMATIC_SALVAGE_ENABLE", "1"))
GLYPHMATIC_SALVAGE_FORCE = int(os.environ.get("GLYPHMATIC_SALVAGE_FORCE", "0"))
GLYPHMATIC_SALVAGE_HEAD_RANK = int(os.environ.get("GLYPHMATIC_SALVAGE_HEAD_RANK", str(GLYPHMATIC_HEAD_RANK)))
GLYPHMATIC_SALVAGE_LOCAL_RANK = int(os.environ.get("GLYPHMATIC_SALVAGE_LOCAL_RANK", "2"))
GLYPHMATIC_SALVAGE_PAIR_RANK = int(os.environ.get("GLYPHMATIC_SALVAGE_PAIR_RANK", "2"))
GLYPHMATIC_CLOSEST_MATCH_ENABLE = int(os.environ.get("GLYPHMATIC_CLOSEST_MATCH_ENABLE", "1"))
GLYPHMATIC_CLOSEST_MATCH_RANK = int(os.environ.get("GLYPHMATIC_CLOSEST_MATCH_RANK", "2"))
GLYPHMATIC_SALVAGE_MARGIN = float(os.environ.get("GLYPHMATIC_SALVAGE_MARGIN", "0.0025"))

GLYPHMATIC_BLOCK_SCORE_WEIGHT = float(os.environ.get("GLYPHMATIC_BLOCK_SCORE_WEIGHT", "0.35"))
GLYPHMATIC_ENERGY_SCORE_WEIGHT = float(os.environ.get("GLYPHMATIC_ENERGY_SCORE_WEIGHT", "0.15"))
GLYPHMATIC_GLOBAL_DRIFT_LIMIT = float(os.environ.get("GLYPHMATIC_GLOBAL_DRIFT_LIMIT", "0.006"))


def hy_float_item(x: Any) -> float:
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    return float(x)


def hy_as_float_matrix(x: torch.Tensor) -> torch.Tensor:
    return x.detach().to(dtype=torch.float32)


def hy_safe_norm(x: torch.Tensor) -> torch.Tensor:
    return torch.linalg.vector_norm(x.float()).clamp_min(torch.tensor(1e-12, device=x.device))


def hy_svd(x: torch.Tensor):
    return torch.linalg.svd(x.float(), full_matrices=False)


def hy_delta_from_lora(merged_lora_B: torch.Tensor, merged_lora_A: torch.Tensor) -> torch.Tensor:
    return hy_as_float_matrix(merged_lora_B) @ hy_as_float_matrix(merged_lora_A)


def hy_reconstruct(B: torch.Tensor, A_rows: torch.Tensor) -> torch.Tensor:
    return B.float() @ A_rows.float()


def hy_candidate_record(name: str, B: torch.Tensor, A_rows: torch.Tensor, delta: torch.Tensor, comp_slices, stats: Dict[str, Any]) -> Dict[str, Any]:
    recon = hy_reconstruct(B, A_rows)
    score = hy_balanced_rebuild_score(delta, recon, comp_slices)
    return {"name": name, "B": B, "A": A_rows, "score": score, "stats": {**stats, "candidate": name, "score": score}}


def hy_factor_from_orthonormal_right_basis(
    delta: torch.Tensor,
    right_basis_rows: torch.Tensor,
    *,
    original_dtype: torch.dtype,
    original_device: torch.device,
    head_rank: int,
    tail_factor: float,
    restore_ratio: float,
    scale_hard_cap: float,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    """Refit B = Delta @ A.T for QR-orthonormal A rows, then apply tailguard/global restore."""
    A_rows = right_basis_rows.float()
    B_cols = delta.float() @ A_rows.T

    r = A_rows.shape[0]
    h = max(0, min(int(head_rank), r))
    if tail_factor < 1.0 and h < r:
        B_cols[:, h:] = B_cols[:, h:] * float(tail_factor)

    target_norm = hy_safe_norm(delta) * float(restore_ratio)
    current_norm = hy_safe_norm(B_cols)
    scale = target_norm / current_norm
    raw_scale = hy_float_item(scale)
    if scale_hard_cap and scale_hard_cap > 0:
        scale = torch.clamp(scale, max=float(scale_hard_cap))

    B_cols = B_cols * scale

    stats = {
        "right_basis_rank": int(r),
        "head_rank": int(h),
        "tail_factor": float(tail_factor),
        "restore_ratio": float(restore_ratio),
        "raw_restore_scale": raw_scale,
        "effective_restore_scale": hy_float_item(scale),
    }

    return (
        B_cols.to(device=original_device, dtype=original_dtype).contiguous(),
        A_rows.to(device=original_device, dtype=original_dtype).contiguous(),
        stats,
    )

# ------------------------------
# Source candidate 1: tailguard / 100% restore
# ------------------------------

def hy_global_tailguard_baseline(
    delta: torch.Tensor,
    rank: int,
    *,
    original_dtype: torch.dtype,
    original_device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    U, S, Vh = hy_svd(delta)
    r = min(int(rank), int(S.numel()))
    S_r = S[:r]
    Vh_r = Vh[:r, :]

    B, A_rows, restore_stats = hy_factor_from_orthonormal_right_basis(
        delta,
        Vh_r,
        original_dtype=original_dtype,
        original_device=original_device,
        head_rank=GLYPHMATIC_HEAD_RANK,
        tail_factor=GLYPHMATIC_TAIL_FACTOR,
        restore_ratio=GLYPHMATIC_RESTORE_RATIO,
        scale_hard_cap=GLYPHMATIC_SCALE_HARD_CAP,
    )

    total_energy = hy_float_item((S ** 2).sum().clamp_min(1e-12))
    kept_energy = hy_float_item((S_r ** 2).sum())
    stats = {
        "method": "source_tailguard_global_svd_100_restore",
        "rank_in": int(delta.shape[0]),
        "rank_out": int(r),
        "full_svd_rank": int(S.numel()),
        "svd_energy_kept_ratio_before_restore": kept_energy / max(total_energy, 1e-12),
        "preservation": "rank32_glyphmatic_frobenius_100_restore_tailguard",
        **restore_stats,
    }
    return B, A_rows, stats

# ------------------------------
# Source candidate 2: dual-pair energy cap
# ------------------------------

def hy_rank_pair(rank: int) -> Tuple[int, int]:
    first = rank // 2
    second = rank - first
    return first, second


def hy_dual_lane_gain_vector(*, singular_values: torch.Tensor, raw_gain: torch.Tensor, rank: int) -> Tuple[torch.Tensor, Dict[str, Any]]:
    base_gain = torch.clamp(raw_gain, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))
    gain_vec = torch.ones_like(singular_values) * base_gain

    if not DUAL_PAIR_ENABLED or rank < 2:
        return gain_vec, {
            "mode": "single_lane_energy_cap",
            "raw_energy_gain": hy_float_item(raw_gain),
            "global_energy_gain": hy_float_item(base_gain),
            "lane_gains": [hy_float_item(base_gain)],
            "lane_caps": [float(SVD_ENERGY_GAIN_CAP)],
            "lane_ranks": [int(rank)],
        }

    r0, r1 = hy_rank_pair(rank)
    pair_mean = sum(DUAL_PAIR_SPLIT) / 2.0
    cap_excess = max(float(SVD_ENERGY_GAIN_CAP) - 1.0, 0.0)
    lane_caps = [
        min(float(SVD_ENERGY_GAIN_CAP), 1.0 + cap_excess * (DUAL_PAIR_SPLIT[0] / pair_mean)),
        min(float(SVD_ENERGY_GAIN_CAP), 1.0 + cap_excess * (DUAL_PAIR_SPLIT[1] / pair_mean)),
    ]

    lane_gain_0 = torch.clamp(raw_gain, min=1.0, max=lane_caps[0])
    lane_gain_1 = torch.clamp(raw_gain, min=1.0, max=lane_caps[1])
    gain_vec[:r0] = lane_gain_0
    gain_vec[r0:r0 + r1] = lane_gain_1

    return gain_vec, {
        "mode": "dual_pair_rank_split_tail_guarded",
        "raw_energy_gain": hy_float_item(raw_gain),
        "global_energy_gain_cap": float(SVD_ENERGY_GAIN_CAP),
        "dual_pair_split": [float(x) for x in DUAL_PAIR_SPLIT],
        "lane_ranks": [int(r0), int(r1)],
        "lane_caps": [float(x) for x in lane_caps],
        "lane_gains": [hy_float_item(lane_gain_0), hy_float_item(lane_gain_1)],
    }


def hy_dual_pair_energy_compress(
    delta: torch.Tensor,
    rank: int,
    *,
    original_dtype: torch.dtype,
    original_device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    U, S, Vh = hy_svd(delta)
    r = min(int(rank), int(S.numel()))
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U_r = U[:, :r]
    S_r = S[:r]
    Vh_r = Vh[:r, :]

    kept_energy = torch.sqrt(torch.sum(S_r ** 2)).clamp_min(1e-12)
    raw_gain = full_energy / kept_energy
    gain_vec, gain_stats = hy_dual_lane_gain_vector(singular_values=S_r, raw_gain=raw_gain, rank=r)

    sroot = torch.sqrt(S_r.clamp_min(1e-30))
    B_new = (U_r * sroot.unsqueeze(0)) * gain_vec.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * Vh_r

    restored_energy = torch.sqrt(torch.sum((S_r * gain_vec) ** 2)).clamp_min(1e-12)
    stats = {
        "method": "source_dual_pair_svd_energy_cap",
        "rank_in": int(delta.shape[0]),
        "rank_out": int(r),
        "singular_mass_kept": hy_float_item(S_r.sum() / total_mass),
        "energy_kept_ratio": hy_float_item(kept_energy / full_energy),
        "energy_after_gain_ratio": hy_float_item(restored_energy / full_energy),
        "preservation": "rank32_dual_pair_svd_energy_restored",
        **gain_stats,
    }
    return B_new.to(device=original_device, dtype=original_dtype).contiguous(), A_new.to(device=original_device, dtype=original_dtype).contiguous(), stats

# ------------------------------
# Source candidate 3: deterministic residual salvage
# ------------------------------

def hy_orthonormalize_rows(candidate_rows: List[torch.Tensor], max_rank: int) -> torch.Tensor:
    rows = []
    for x in candidate_rows:
        if x is None:
            continue
        if x.ndim == 1:
            x = x.unsqueeze(0)
        if x.numel() == 0:
            continue
        rows.append(x.float())
    if not rows:
        raise RuntimeError("No candidate basis rows supplied for salvage rebuild")

    C = torch.cat(rows, dim=0)
    C = C[torch.isfinite(C).all(dim=1)]
    if C.shape[0] == 0:
        raise RuntimeError("Candidate basis rows were non-finite")

    n = torch.linalg.vector_norm(C, dim=1)
    C = C[n > 1e-8]
    if C.shape[0] == 0:
        raise RuntimeError("Candidate basis rows were zero")

    Q, _ = torch.linalg.qr(C.T.contiguous(), mode="reduced")
    r = min(int(max_rank), Q.shape[1])
    return Q[:, :r].T.contiguous()


def hy_row_pair_ranges(comp_slices):
    ranges = []
    if len(comp_slices) >= 2:
        for i in range(0, len(comp_slices) - 1, 2):
            s0 = comp_slices[i][0]
            e1 = comp_slices[i + 1][1]
            names = f"{comp_slices[i][3]}+{comp_slices[i + 1][3]}"
            ranges.append((s0, e1, names))
    if len(comp_slices) > 2:
        ranges.append((comp_slices[0][0], comp_slices[-1][1], "all_components"))
    return ranges


def hy_normalized_rows(rows: torch.Tensor) -> torch.Tensor:
    return rows.float() / torch.linalg.vector_norm(rows.float(), dim=1, keepdim=True).clamp_min(1e-12)


def hy_closest_match_rows(local_rows: torch.Tensor, global_tail_rows: torch.Tensor, max_rows: int) -> Tuple[torch.Tensor, Dict[str, Any]]:
    """Deterministically blend residual-local directions with closest discarded global directions."""
    if max_rows <= 0 or local_rows.numel() == 0 or global_tail_rows.numel() == 0:
        return torch.empty((0, local_rows.shape[-1] if local_rows.ndim == 2 else 0), device=local_rows.device), {"rank": 0}

    L = hy_normalized_rows(local_rows)
    G = hy_normalized_rows(global_tail_rows)
    sim = torch.abs(L @ G.T)
    flat = torch.argsort(sim.flatten(), descending=True)

    used_l = set()
    used_g = set()
    chosen = []
    pairs = []
    for idx in flat:
        i = int((idx // sim.shape[1]).detach().cpu())
        j = int((idx % sim.shape[1]).detach().cpu())
        if i in used_l or j in used_g:
            continue
        blended = 0.70 * L[i] + 0.30 * G[j]
        blended = blended / torch.linalg.vector_norm(blended).clamp_min(1e-12)
        chosen.append(blended.unsqueeze(0))
        pairs.append({"local_row": i, "global_tail_row": j, "abs_cosine": hy_float_item(sim[i, j])})
        used_l.add(i)
        used_g.add(j)
        if len(chosen) >= max_rows:
            break

    if not chosen:
        return torch.empty((0, local_rows.shape[1]), device=local_rows.device), {"rank": 0}
    return torch.cat(chosen, dim=0), {"rank": len(chosen), "pairs": pairs}


def hy_component_residual_basis(
    delta: torch.Tensor,
    head_recon: torch.Tensor,
    comp_slices,
    *,
    local_rank: int,
    pair_rank: int,
    global_tail_rows: torch.Tensor,
    closest_rank: int,
) -> Tuple[List[torch.Tensor], Dict[str, Any]]:
    basis_rows: List[torch.Tensor] = []
    details: Dict[str, Any] = {"component_basis": [], "pair_basis": [], "closest_match_basis": []}
    residual = delta.float() - head_recon.float()

    local_direction_batches: List[torch.Tensor] = []

    if local_rank > 0:
        for row_start, row_end, _r, comp_name in comp_slices:
            block = residual[row_start:row_end, :]
            if block.numel() == 0 or hy_float_item(hy_safe_norm(block)) <= 1e-8:
                continue
            try:
                _U, S, Vh = hy_svd(block)
                k = min(int(local_rank), int(S.numel()))
                if k > 0:
                    rows = Vh[:k, :]
                    basis_rows.append(rows)
                    local_direction_batches.append(rows)
                    details["component_basis"].append({
                        "component": str(comp_name),
                        "rows": [int(row_start), int(row_end)],
                        "rank": int(k),
                        "residual_norm": hy_float_item(hy_safe_norm(block)),
                    })
            except Exception as e:
                details["component_basis"].append({
                    "component": str(comp_name),
                    "rows": [int(row_start), int(row_end)],
                    "rank": 0,
                    "error": repr(e)[:180],
                })

    if pair_rank > 0:
        for row_start, row_end, names in hy_row_pair_ranges(comp_slices):
            block = residual[row_start:row_end, :]
            if block.numel() == 0 or hy_float_item(hy_safe_norm(block)) <= 1e-8:
                continue
            try:
                _U, S, Vh = hy_svd(block)
                k = min(int(pair_rank), int(S.numel()))
                if k > 0:
                    rows = Vh[:k, :]
                    basis_rows.append(rows)
                    local_direction_batches.append(rows)
                    details["pair_basis"].append({
                        "pair": str(names),
                        "rows": [int(row_start), int(row_end)],
                        "rank": int(k),
                        "residual_norm": hy_float_item(hy_safe_norm(block)),
                    })
            except Exception as e:
                details["pair_basis"].append({
                    "pair": str(names),
                    "rows": [int(row_start), int(row_end)],
                    "rank": 0,
                    "error": repr(e)[:180],
                })

    if GLYPHMATIC_CLOSEST_MATCH_ENABLE and closest_rank > 0 and local_direction_batches:
        local_rows = torch.cat(local_direction_batches, dim=0)
        try:
            close_rows, close_stats = hy_closest_match_rows(local_rows, global_tail_rows, closest_rank)
            if close_rows.numel() > 0:
                basis_rows.append(close_rows)
            details["closest_match_basis"].append(close_stats)
        except Exception as e:
            details["closest_match_basis"].append({"rank": 0, "error": repr(e)[:180]})

    return basis_rows, details


def hy_balanced_rebuild_score(delta: torch.Tensor, recon: torch.Tensor, comp_slices) -> Dict[str, Any]:
    full_norm = hy_safe_norm(delta)
    global_rel_error = hy_float_item(hy_safe_norm(delta - recon) / full_norm)

    block_errors = []
    block_energy_errors = []
    block_stats = []

    for row_start, row_end, _r, comp_name in comp_slices:
        d = delta[row_start:row_end, :].float()
        q = recon[row_start:row_end, :].float()
        dn = hy_safe_norm(d)
        rel_error = hy_float_item(hy_safe_norm(d - q) / dn)
        energy_ratio = hy_float_item(hy_safe_norm(q) / dn)
        energy_error = abs(1.0 - energy_ratio)

        block_errors.append(rel_error)
        block_energy_errors.append(energy_error)
        block_stats.append({
            "component": str(comp_name),
            "rows": [int(row_start), int(row_end)],
            "rel_error": rel_error,
            "energy_ratio": energy_ratio,
        })

    mean_block_error = float(sum(block_errors) / max(len(block_errors), 1))
    max_block_error = float(max(block_errors) if block_errors else 0.0)
    mean_energy_error = float(sum(block_energy_errors) / max(len(block_energy_errors), 1))

    score = (
        global_rel_error
        + GLYPHMATIC_BLOCK_SCORE_WEIGHT * max_block_error
        + GLYPHMATIC_ENERGY_SCORE_WEIGHT * mean_energy_error
    )

    return {
        "score": float(score),
        "global_rel_error": global_rel_error,
        "mean_block_rel_error": mean_block_error,
        "max_block_rel_error": max_block_error,
        "mean_block_energy_abs_error": mean_energy_error,
        "blocks": block_stats,
    }


def hy_salvage_rebuild_candidate(
    delta: torch.Tensor,
    rank: int,
    comp_slices,
    *,
    original_dtype: torch.dtype,
    original_device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    U, S, Vh = hy_svd(delta)
    head_rank = min(max(1, GLYPHMATIC_SALVAGE_HEAD_RANK), int(rank), int(S.numel()))
    head_U = U[:, :head_rank]
    head_S = S[:head_rank]
    head_Vh = Vh[:head_rank, :]
    head_recon = (head_U * head_S.unsqueeze(0)) @ head_Vh

    candidate_rows: List[torch.Tensor] = [head_Vh]
    tail_rows = Vh[head_rank:, :]

    local_rows, local_details = hy_component_residual_basis(
        delta,
        head_recon,
        comp_slices,
        local_rank=GLYPHMATIC_SALVAGE_LOCAL_RANK,
        pair_rank=GLYPHMATIC_SALVAGE_PAIR_RANK,
        global_tail_rows=tail_rows,
        closest_rank=GLYPHMATIC_CLOSEST_MATCH_RANK,
    )
    candidate_rows.extend(local_rows)

    # Deterministic coverage fill: keep extra global tail rows so local salvage cannot collapse rank coverage.
    if tail_rows.numel() > 0:
        candidate_rows.append(tail_rows[:min(max(0, int(rank) + 16 - head_rank), tail_rows.shape[0]), :])

    A_rows = hy_orthonormalize_rows(candidate_rows, max_rank=rank)
    salv_B, salv_A, restore_stats = hy_factor_from_orthonormal_right_basis(
        delta,
        A_rows,
        original_dtype=original_dtype,
        original_device=original_device,
        head_rank=GLYPHMATIC_HEAD_RANK,
        tail_factor=GLYPHMATIC_TAIL_FACTOR,
        restore_ratio=GLYPHMATIC_RESTORE_RATIO,
        scale_hard_cap=GLYPHMATIC_SCALE_HARD_CAP,
    )

    stats = {
        "method": "glyphmatic_deterministic_component_residual_salvage_rebuild_v2",
        "rank_in": int(delta.shape[0]),
        "rank_out": int(salv_A.shape[0]),
        "head_rank": int(head_rank),
        "salvage_local_rank": int(GLYPHMATIC_SALVAGE_LOCAL_RANK),
        "salvage_pair_rank": int(GLYPHMATIC_SALVAGE_PAIR_RANK),
        "closest_match_enable": int(GLYPHMATIC_CLOSEST_MATCH_ENABLE),
        "closest_match_rank": int(GLYPHMATIC_CLOSEST_MATCH_RANK),
        "basis_details": local_details,
        "preservation": "rank32_component_residual_closest_match_pair_of_pairs_tailguard",
        **restore_stats,
    }
    return salv_B, salv_A, stats

# ------------------------------
# Candidate selector
# ------------------------------

def hy_candidate_allowed(name: str) -> bool:
    mode = GLYPHMATIC_PATCH_MODE
    if GLYPHMATIC_FORCE_CANDIDATE:
        return name == GLYPHMATIC_FORCE_CANDIDATE
    if mode in {"tailguard", "baseline", "global"}:
        return name == "tailguard"
    if mode in {"dual_pair", "dual", "energy_cap"}:
        return name == "dual_pair"
    if mode in {"salvage", "salvage_guarded"}:
        return name in {"tailguard", "salvage"}
    return name in {"tailguard", "dual_pair", "salvage"}


def hy_select_best_candidate(candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not candidates:
        raise RuntimeError("No compression candidates were produced")

    by_name = {c["name"]: c for c in candidates}
    if GLYPHMATIC_FORCE_CANDIDATE and GLYPHMATIC_FORCE_CANDIDATE in by_name:
        forced = by_name[GLYPHMATIC_FORCE_CANDIDATE]
        forced["stats"] = {**forced["stats"], "selected": forced["name"], "selection_reason": "forced_env"}
        return forced

    baseline = by_name.get("tailguard", candidates[0])
    best = min(candidates, key=lambda c: c["score"]["score"])

    # Guard: reject non-tailguard candidates that only win by a tiny/noisy margin and increase global error too much.
    if best["name"] != "tailguard":
        margin_win = baseline["score"]["score"] - best["score"]["score"]
        global_drift = best["score"]["global_rel_error"] - baseline["score"]["global_rel_error"]
        forced_salvage = bool(GLYPHMATIC_SALVAGE_FORCE and best["name"] == "salvage")
        enough_margin = margin_win >= GLYPHMATIC_SALVAGE_MARGIN
        block_improved = best["score"]["max_block_rel_error"] < baseline["score"]["max_block_rel_error"] * 0.985
        drift_ok = global_drift <= GLYPHMATIC_GLOBAL_DRIFT_LIMIT

        if not forced_salvage and not (enough_margin or (block_improved and drift_ok)):
            selected = baseline
            reason = "baseline_guard_margin"
        elif not forced_salvage and not drift_ok:
            selected = baseline
            reason = "baseline_guard_global_drift"
        else:
            selected = best
            reason = "best_balanced_score"
    else:
        selected = best
        reason = "tailguard_best"

    selected["stats"] = {
        **selected["stats"],
        "selected": selected["name"],
        "selection_reason": reason,
        "selected_score": selected["score"]["score"],
        "all_candidate_scores": {
            c["name"]: {
                "score": c["score"]["score"],
                "global_rel_error": c["score"]["global_rel_error"],
                "max_block_rel_error": c["score"]["max_block_rel_error"],
                "mean_block_energy_abs_error": c["score"]["mean_block_energy_abs_error"],
            }
            for c in candidates
        },
    }
    return selected


def hy_hybrid_compress(
    merged_lora_B: torch.Tensor,
    merged_lora_A: torch.Tensor,
    rank: int,
    comp_slices,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    original_dtype = merged_lora_A.dtype
    original_device = merged_lora_A.device
    delta = hy_delta_from_lora(merged_lora_B, merged_lora_A)
    candidates: List[Dict[str, Any]] = []

    if hy_candidate_allowed("tailguard"):
        try:
            B, A_rows, stats = hy_global_tailguard_baseline(delta, rank, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record("tailguard", B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print("[GlyphMatics] tailguard candidate failed:", repr(e)[:240])

    if hy_candidate_allowed("dual_pair"):
        try:
            B, A_rows, stats = hy_dual_pair_energy_compress(delta, rank, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record("dual_pair", B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print("[GlyphMatics] dual_pair candidate failed:", repr(e)[:240])

    if GLYPHMATIC_SALVAGE_ENABLE and hy_candidate_allowed("salvage"):
        try:
            B, A_rows, stats = hy_salvage_rebuild_candidate(delta, rank, comp_slices, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record("salvage", B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print("[GlyphMatics] salvage candidate failed:", repr(e)[:240])

    if not candidates:
        # Final defensive fallback if the user forced a bad candidate name.
        B, A_rows, stats = hy_global_tailguard_baseline(delta, rank, original_dtype=original_dtype, original_device=original_device)
        record = hy_candidate_record("tailguard", B, A_rows, delta, comp_slices, {**stats, "fallback_reason": "no_candidates"})
        candidates.append(record)

    selected = hy_select_best_candidate(candidates)
    return selected["B"], selected["A"], selected["stats"]





# -----------------------------------------------------------------------------
# PairFold candidate wrapper + hybrid selector
# -----------------------------------------------------------------------------

def _pairfold_candidate_record(
    merged_lora_B: torch.Tensor,
    merged_lora_A: torch.Tensor,
    rank: int,
    comp_slices,
):
    delta = hy_delta_from_lora(merged_lora_B, merged_lora_A)
    B, A_rows, stats = _compress_lora_pair_to_rank(merged_lora_B, merged_lora_A, rank, component_slices=comp_slices)
    score = hy_balanced_rebuild_score(delta, hy_reconstruct(B, A_rows), comp_slices)
    stats = {
        **stats,
        'method': 'source_uploaded_v8_pairfold_rowguard',
    }
    return {
        'name': 'pairfold',
        'B': B,
        'A': A_rows,
        'stats': stats,
        'score': score,
    }


def glyphmatic_hybrid_compress(
    merged_lora_B: torch.Tensor,
    merged_lora_A: torch.Tensor,
    rank: int,
    comp_slices,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
    original_dtype = merged_lora_A.dtype
    original_device = merged_lora_A.device
    delta = hy_delta_from_lora(merged_lora_B, merged_lora_A)
    candidates: List[Dict[str, Any]] = []

    force = GLYPHMATIC_FORCE_CANDIDATE.strip().lower()

    # v11: strict force mode. If a candidate is forced, do not let the saturated
    # baseline/tailguard candidate silently win again.
    allow_tailguard = (force in {'', 'tailguard'})
    allow_pairfold = (force in {'', 'pairfold'})
    allow_dual_pair = (force in {'', 'dual_pair'})
    allow_salvage = (force in {'', 'salvage'})

    if allow_tailguard and hy_candidate_allowed('tailguard'):
        try:
            B, A_rows, stats = hy_global_tailguard_baseline(delta, rank, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record('tailguard', B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print('[GlyphMatics] tailguard candidate failed:', repr(e)[:240])

    # Uploaded notebook base candidate.
    if allow_pairfold:
        try:
            candidates.append(_pairfold_candidate_record(merged_lora_B, merged_lora_A, rank, comp_slices))
        except Exception as e:
            print('[GlyphMatics] pairfold candidate failed:', repr(e)[:240])

    if allow_dual_pair and hy_candidate_allowed('dual_pair'):
        try:
            B, A_rows, stats = hy_dual_pair_energy_compress(delta, rank, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record('dual_pair', B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print('[GlyphMatics] dual_pair candidate failed:', repr(e)[:240])

    if allow_salvage and GLYPHMATIC_SALVAGE_ENABLE:
        try:
            B, A_rows, stats = hy_salvage_rebuild_candidate(delta, rank, comp_slices, original_dtype=original_dtype, original_device=original_device)
            candidates.append(hy_candidate_record('salvage', B, A_rows, delta, comp_slices, stats))
        except Exception as e:
            print('[GlyphMatics] salvage candidate failed:', repr(e)[:240])

    if not candidates:
        # Always keep a defensive fallback.
        B, A_rows, stats = hy_global_tailguard_baseline(delta, rank, original_dtype=original_dtype, original_device=original_device)
        candidates.append(hy_candidate_record('tailguard', B, A_rows, delta, comp_slices, {**stats, 'fallback_reason': 'no_candidates'}))

    selected = hy_select_best_candidate(candidates)
    selected['stats'] = {
        **selected['stats'],
        'selected': selected['name'],
        'transport': 'glyphmatic_forced_salvage_rebuild_v11',
    }
    return selected['B'], selected['A'], selected['stats']


# -----------------------------------------------------------------------------
# Tinker patch: fused projection merge
# -----------------------------------------------------------------------------
def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix('.weight').rsplit('.', 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    if component_order is None:
        raise RuntimeError(f'No fused projection component order found for {fused_target_name!r}')

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f'Missing component {comp_name!r} for fused target {fused_model_key!r}')
        lora_A, lora_B = comp_by_name[comp_name]
        if lora_A.ndim != 2 or lora_B.ndim != 2:
            raise ValueError(f'Expected 2D LoRA tensors for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}')
        if lora_A.shape[0] != lora_B.shape[1]:
            raise ValueError(f'Component rank mismatch for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}')
        r = int(lora_A.shape[0])
        out_dim = int(lora_B.shape[0])
        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device)

    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = int(merged_rank)
    compression_stats = {
        'rank_in': int(merged_rank),
        'rank_out': int(merged_rank),
        'preservation': 'exact_no_compression',
        'selected': 'exact',
        'component_row_total': int(row_offset),
        'fused_out_dim': int(fused_out_dim),
    }

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, compression_stats = glyphmatic_hybrid_compress(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK,
            comp_slices,
        )
        final_rank = int(merged_lora_A.shape[0])
        compression_stats = {
            **compression_stats,
            'rank_in': int(merged_rank),
            'rank_out': int(final_rank),
            'gain_cap': float(SVD_ENERGY_GAIN_CAP),
        }

    peft_target_key = f'{adapter_layer_prefix}.{fused_target_name}.weight'

    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op='glyphmatic_fused_projection_transport',
        alpha='pairfold_hybrid_components_v9',
        beta={
            'fused_model_key': fused_model_key,
            'fused_out_dim': int(fused_out_dim),
            'component_count': len(component_order),
            'component_order': list(component_order),
            'component_slices': [
                {'name': name, 'rows': [int(a), int(b)], 'rank': int(r)}
                for a, b, r, name in comp_slices
            ],
        },
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank

if not hasattr(A, '_merge_fused_projections'):
    raise RuntimeError('tinker_cookbook.weights._adapter._merge_fused_projections not found')
A._merge_fused_projections = patched_merge_fused_projections

print('[GlyphMatics] patched:', A._merge_fused_projections.__name__)
print('[GlyphMatics] FORCED_FUSED_RANK:', FORCED_FUSED_RANK)
print('[GlyphMatics] PATCH_MODE:', GLYPHMATIC_PATCH_MODE)
print('[GlyphMatics] FORCE_CANDIDATE:', GLYPHMATIC_FORCE_CANDIDATE or '<none>')
print('[GlyphMatics] pairfold:', {'enabled': PAIRFOLD_ENABLED, 'row_guard': ROW_NORM_GUARD_ENABLED})
print('[GlyphMatics] tailguard:', {'restore': GLYPHMATIC_RESTORE_RATIO, 'head': GLYPHMATIC_HEAD_RANK, 'tail': GLYPHMATIC_TAIL_FACTOR})
print('[GlyphMatics] dual_pair:', {'cap': SVD_ENERGY_GAIN_CAP, 'enabled': DUAL_PAIR_ENABLED, 'split': DUAL_PAIR_SPLIT})
print('[GlyphMatics] salvage:', {'enabled': GLYPHMATIC_SALVAGE_ENABLE, 'head': GLYPHMATIC_SALVAGE_HEAD_RANK, 'local': GLYPHMATIC_SALVAGE_LOCAL_RANK, 'pair': GLYPHMATIC_SALVAGE_PAIR_RANK, 'closest': GLYPHMATIC_CLOSEST_MATCH_RANK})
print('[GlyphMatics] transport: glyphmatic_forced_salvage_rebuild_v11')


def glyphmatic_patch_selftest() -> None:
    """Fast deterministic tensor test; exercises the uploaded base candidate and the new hybrid selector."""
    torch.manual_seed(918)
    dtype = torch.float32
    B1 = torch.randn(8, 4, dtype=dtype) * 0.08
    A1 = torch.randn(4, 12, dtype=dtype) * 0.08
    B2 = torch.randn(6, 4, dtype=dtype) * 0.08
    A2 = torch.randn(4, 12, dtype=dtype) * 0.08
    B3 = torch.randn(5, 4, dtype=dtype) * 0.05
    A3 = torch.randn(4, 12, dtype=dtype) * 0.05
    B4 = torch.randn(7, 4, dtype=dtype) * 0.05
    A4 = torch.randn(4, 12, dtype=dtype) * 0.05

    merged_A = torch.cat([A1, A2, A3, A4], dim=0)
    merged_B = torch.zeros(26, 16, dtype=dtype)
    merged_B[0:8, 0:4] = B1
    merged_B[8:14, 4:8] = B2
    merged_B[14:19, 8:12] = B3
    merged_B[19:26, 12:16] = B4
    comp_slices = [(0, 8, 4, 'test_a'), (8, 14, 4, 'test_b'), (14, 19, 4, 'test_c'), (19, 26, 4, 'test_d')]

    # Base uploaded PairFold candidate stays operational.
    out_B0, out_A0, stats0 = _compress_lora_pair_to_rank(merged_B, merged_A, 8, component_slices=comp_slices)
    assert out_A0.shape[0] == 8, out_A0.shape
    assert out_B0.shape[1] == 8, out_B0.shape

    # New hybrid selector.
    out_B, out_A, stats = glyphmatic_hybrid_compress(merged_B, merged_A, 8, comp_slices)
    assert out_A.shape[0] <= 8, out_A.shape
    assert out_B.shape[1] == out_A.shape[0], (out_B.shape, out_A.shape)
    assert torch.isfinite(out_A.float()).all(), 'A contains non-finite values'
    assert torch.isfinite(out_B.float()).all(), 'B contains non-finite values'
    assert stats.get('selected') in {'salvage', 'tailguard'}, stats.get('selected')  # tailguard only if salvage hard-fails
    print('[GlyphMatics selftest] PASS')
    print(json.dumps({
        'pairfold_rank_out': stats0.get('rank_out'),
        'selected': stats.get('selected'),
        'reason': stats.get('selection_reason'),
        'rank_out': stats.get('rank_out'),
        'scores': stats.get('all_candidate_scores'),
    }, indent=2, sort_keys=True))

glyphmatic_patch_selftest()


[GlyphMatics] patched: patched_merge_fused_projections
[GlyphMatics] FORCED_FUSED_RANK: 32
[GlyphMatics] PATCH_MODE: salvage_guarded
[GlyphMatics] FORCE_CANDIDATE: salvage
[GlyphMatics] pairfold: {'enabled': True, 'row_guard': True}
[GlyphMatics] tailguard: {'restore': 1.0, 'head': 24, 'tail': 0.88}
[GlyphMatics] dual_pair: {'cap': 1.2, 'enabled': True, 'split': [0.62, 0.58]}
[GlyphMatics] salvage: {'enabled': 1, 'head': 24, 'local': 3, 'pair': 3, 'closest': 4}
[GlyphMatics] transport: glyphmatic_forced_salvage_rebuild_v11
[GlyphMatics selftest] PASS
{
  "pairfold_rank_out": 8,
  "rank_out": 8,
  "reason": "forced_env",
  "scores": null,
  "selected": "salvage"
}


## 4. Local compression self-tests

Fast deterministic tensor tests to verify that both the uploaded PairFold base path and the new hybrid selector are working before the full build.


In [5]:
# The main patch cell already runs glyphmatic_patch_selftest().
# Re-run on demand here for convenience.
glyphmatic_patch_selftest()


[GlyphMatics selftest] PASS
{
  "pairfold_rank_out": 8,
  "rank_out": 8,
  "reason": "forced_env",
  "scores": null,
  "selected": "salvage"
}


## 5. Build adapter package

This uses the cookbook builder with a tolerant signature adapter, writes a manifest, and persists the transport ledger.


In [6]:
from pathlib import Path
import shutil
import json
import time
import os
import inspect
from tinker_cookbook import weights

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working/nemotron_glyphmatics_submission_model"))
ZIP_PATH = Path(os.environ.get("ZIP_PATH", "/kaggle/working/submission.zip"))
LEDGER_PATH = Path("/kaggle/working/glyphmatic_transport_ledger.md")
MANIFEST_PATH = Path("/kaggle/working/submission_manifest.json")

# Tinker's build_lora_adapter expects output_path to NOT exist.
# Therefore: remove stale output, but do not recreate OUTPUT_DIR before calling the builder.
for p in [OUTPUT_DIR, ZIP_PATH, LEDGER_PATH, MANIFEST_PATH]:
    if p.exists():
        print("[Build] removing stale path:", p)
        if p.is_dir():
            shutil.rmtree(p)
        else:
            p.unlink()

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

print("[Build] output_dir:", OUTPUT_DIR)
print("[Build] zip_path:", ZIP_PATH)
print("[Build] build_lora_adapter:", inspect.signature(weights.build_lora_adapter))


def _call_build_lora_adapter():
    # Call the cookbook builder while tolerating minor argument-name drift.
    fn = weights.build_lora_adapter
    sig = inspect.signature(fn)
    params = set(sig.parameters)
    kwargs = {}

    for name in ["base_model", "base_model_path", "model_path"]:
        if name in params:
            kwargs[name] = str(BASE_MODEL_PATH)
            break
    for name in ["adapter_path", "lora_path", "adapter"]:
        if name in params:
            kwargs[name] = str(ADAPTER_PATH)
            break
    for name in ["output_path", "output_dir", "save_path"]:
        if name in params:
            kwargs[name] = str(OUTPUT_DIR)
            break

    if len(kwargs) >= 3:
        # Last-moment guard: cookbook raises FileExistsError if output_path already exists.
        if OUTPUT_DIR.exists():
            print("[Build] last-moment removing stale OUTPUT_DIR:", OUTPUT_DIR)
            shutil.rmtree(OUTPUT_DIR)
        return fn(**kwargs)

    # Known cookbook signature fallback.
    if OUTPUT_DIR.exists():
        print("[Build] last-moment removing stale OUTPUT_DIR:", OUTPUT_DIR)
        shutil.rmtree(OUTPUT_DIR)
    return fn(base_model=str(BASE_MODEL_PATH), adapter_path=str(ADAPTER_PATH), output_path=str(OUTPUT_DIR))


t0 = time.time()
build_result = _call_build_lora_adapter()
elapsed = time.time() - t0
print("[Build] complete in %.1f sec" % elapsed)
print("[Build] result:", build_result)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Builder completed but OUTPUT_DIR was not created: {OUTPUT_DIR}")

# Required/defensive completion marker.
(OUTPUT_DIR / "checkpoint_complete").write_text(
    "complete\n"
    f"elapsed_sec={elapsed:.3f}\n"
    f"patch_mode={os.environ.get('GLYPHMATIC_PATCH_MODE')}\n"
    f"forced_candidate={os.environ.get('GLYPHMATIC_FORCE_CANDIDATE', '')}\n",
    encoding="utf-8",
)

if "GLYPH_LEDGER" in globals():
    GLYPH_LEDGER.print_summary()
    LEDGER_PATH.write_text(GLYPH_LEDGER.markdown(), encoding="utf-8")
    print("[Build] ledger:", LEDGER_PATH)
else:
    print("[Build] GLYPH_LEDGER not found")

manifest = {
    "created_by": "918 Technologies / GlyphMatics PairFold Hybrid Components v9",
    "competition": "NVIDIA Nemotron Model Reasoning Challenge",
    "source_notebooks_integrated": [
        "glyphmatics-1.ipynb",
        "nemotron_glyphmatics_salvage_competition_ready.ipynb",
        "glyphmatics_100_restore_tailguard_submission.ipynb",
        "glyphmatics_100_percent_restore_submission.ipynb",
        "glyphmatics_dual_pair_energy_cap_submission.ipynb",
        "glyphmatics_single_lane_120_recovery_submission.ipynb",
        "glyphmatics_v6_rank36_singlelane_118_submission.ipynb",
        "nemotron_cap_119_submission_candidate.ipynb",
        "nemotron_cap_1195_submission_candidate.ipynb",
        "notebook0a85a85ab7_cap_1_2050_next.ipynb",
    ],
    "adapter_path": str(ADAPTER_PATH),
    "base_model_path": str(BASE_MODEL_PATH),
    "output_dir": str(OUTPUT_DIR),
    "zip_path": str(ZIP_PATH),
    "elapsed_sec": elapsed,
    "env": {
        k: os.environ.get(k)
        for k in sorted(os.environ)
        if k.startswith("GLYPHMATIC") or k in {"FORCED_FUSED_RANK", "SVD_ENERGY_GAIN_CAP", "DUAL_PAIR_ENABLED", "DUAL_PAIR_SPLIT"}
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print("[Build] manifest:", MANIFEST_PATH)

print("\n[Build] output tree preview")
files = sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])
for p in files[:140]:
    print(" -", p.relative_to(OUTPUT_DIR), p.stat().st_size)
if len(files) > 140:
    print(" ...", len(files) - 140, "more files")


[Build] output_dir: /kaggle/working/nemotron_glyphmatics_submission_model
[Build] zip_path: /kaggle/working/submission.zip
[Build] build_lora_adapter: (*, base_model: 'str', adapter_path: 'str', output_path: 'str', trust_remote_code: 'bool | None' = None) -> 'None'


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[Build] complete in 635.5 sec
[Build] result: None
[GlyphMatics ledger] events: 23
[GlyphMatics ledger] pairfold_hybrid_components_v9:glyphmatic_fused_projection_transport=23
[Build] ledger: /kaggle/working/glyphmatic_transport_ledger.md
[Build] manifest: /kaggle/working/submission_manifest.json

[Build] output tree preview
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 81


## 6. Validate and create `submission.zip`

This performs defensive verification before zipping the full output directory.


In [7]:
from pathlib import Path
import zipfile
import json
import os

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working/nemotron_glyphmatics_submission_model"))
ZIP_PATH = Path(os.environ.get("ZIP_PATH", "/kaggle/working/submission.zip"))

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"OUTPUT_DIR not found: {OUTPUT_DIR}")

# Defensive validation before zipping.
existing_files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
existing_names = {p.name for p in existing_files}

config_names = {
    "config.json",
    "adapter_config.json",
    "model.safetensors.index.json",
    "pytorch_model.bin.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
}
has_config_like = bool(existing_names & config_names) or any(p.suffix == ".json" for p in existing_files)
has_weight_like = any(
    p.suffix in {".safetensors", ".bin", ".pt"} or p.name.endswith(".safetensors.index.json")
    for p in existing_files
)
has_marker = (OUTPUT_DIR / "checkpoint_complete").exists()

print("[Verify before zip] file_count:", len(existing_files))
print("[Verify before zip] has_config_like:", has_config_like)
print("[Verify before zip] has_weight_like:", has_weight_like)
print("[Verify before zip] has_checkpoint_complete:", has_marker)

if not has_config_like:
    raise RuntimeError("No config-like file found in output. Build may have failed.")
if not has_weight_like:
    raise RuntimeError("No weight-like file found in output. Build may have failed.")
if not has_marker:
    raise RuntimeError("checkpoint_complete marker missing.")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            arcname = p.relative_to(OUTPUT_DIR).as_posix()
            z.write(p, arcname)

print("[Zip] created:", ZIP_PATH)
print("[Zip] size_mb:", ZIP_PATH.stat().st_size / (1024 * 1024))

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()
    print("[Zip] file_count:", len(names))
    print("[Zip] first files:")
    for n in names[:120]:
        print(" -", n)
    if len(names) > 120:
        print(" ...", len(names) - 120, "more files")

    assert "checkpoint_complete" in names, "checkpoint_complete missing from zip"
    assert any(n.endswith(".json") for n in names), "No JSON/config files in zip"
    assert any(n.endswith(".safetensors") or n.endswith(".bin") or n.endswith(".pt") for n in names), "No weight file in zip"

print("\n[READY] Submit this file:")
print(ZIP_PATH)


[Verify before zip] file_count: 3
[Verify before zip] has_config_like: True
[Verify before zip] has_weight_like: True
[Verify before zip] has_checkpoint_complete: True
[Zip] created: /kaggle/working/submission.zip
[Zip] size_mb: 3118.9352989196777
[Zip] file_count: 3
[Zip] first files:
 - adapter_config.json
 - adapter_model.safetensors
 - checkpoint_complete

[READY] Submit this file:
/kaggle/working/submission.zip


## 7. Final listing and fallback profiles

The listing helps confirm what was produced. The fallback profile block below can be edited and rerun from the patch cell onward if you want to force a specific candidate.


In [8]:
from pathlib import Path
import os

for p in [
    Path(os.environ.get("ZIP_PATH", "/kaggle/working/submission.zip")),
    Path(os.environ.get("OUTPUT_DIR", "/kaggle/working/nemotron_glyphmatics_submission_model")),
    Path("/kaggle/working/glyphmatic_transport_ledger.md"),
    Path("/kaggle/working/submission_manifest.json"),
]:
    print("\n[Listing]", p)
    if p.is_file():
        print(p, p.stat().st_size)
    elif p.is_dir():
        count = 0
        for child in sorted(p.rglob("*")):
            if child.is_file():
                print(" -", child.relative_to(p), child.stat().st_size)
                count += 1
                if count >= 80:
                    more = len([x for x in p.rglob("*") if x.is_file()]) - count
                    if more > 0:
                        print(" ...", more, "more files")
                    break
    else:
        print("MISSING:", p)

print("\nFallback examples:")
print(" - tailguard only: os.environ['GLYPHMATIC_FORCE_CANDIDATE']='tailguard'")
print(" - pairfold only : os.environ['GLYPHMATIC_FORCE_CANDIDATE']='pairfold'")
print(" - dual_pair only: os.environ['GLYPHMATIC_FORCE_CANDIDATE']='dual_pair'")
print(" - salvage only  : os.environ['GLYPHMATIC_FORCE_CANDIDATE']='salvage'; os.environ['GLYPHMATIC_SALVAGE_FORCE']='1'")

print(" - v11 default  : forced salvage rebuild; avoids tailguard unless salvage hard-fails")



[Listing] /kaggle/working/submission.zip
/kaggle/working/submission.zip 3270440700

[Listing] /kaggle/working/nemotron_glyphmatics_submission_model
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 81

[Listing] /kaggle/working/glyphmatic_transport_ledger.md
/kaggle/working/glyphmatic_transport_ledger.md 46147

[Listing] /kaggle/working/submission_manifest.json
/kaggle/working/submission_manifest.json 1921

Fallback examples:
 - tailguard only: os.environ['GLYPHMATIC_FORCE_CANDIDATE']='tailguard'
 - pairfold only : os.environ['GLYPHMATIC_FORCE_CANDIDATE']='pairfold'
 - dual_pair only: os.environ['GLYPHMATIC_FORCE_CANDIDATE']='dual_pair'
 - salvage only  : os.environ['GLYPHMATIC_FORCE_CANDIDATE']='salvage'; os.environ['GLYPHMATIC_SALVAGE_FORCE']='1'
 - v11 default  : forced salvage rebuild; avoids tailguard unless salvage hard-fails
